# Tesseract Baseline

Use this notebook only to orchestrate Tesseract baseline runs.
Import reusable logic from `src/`; do not implement business logic here.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
DEFAULT_COLAB_ROOT = Path('/content/nfse-extractor/nfse-extractor')
DEFAULT_LOCAL_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
ROOT = Path(os.environ.get('NFSE_PROJECT_ROOT', DEFAULT_COLAB_ROOT if IS_COLAB else DEFAULT_LOCAL_ROOT))

if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}. Clone the repo into /content/nfse-extractor first or set NFSE_PROJECT_ROOT.'
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

CONFIG = {
    'bootstrap_runtime': True,
    'mount_drive': False,
    'sample_path': os.environ.get('NFSE_SAMPLE_PATH', ''),
    'language': os.environ.get('NFSE_TESSERACT_LANGUAGE', 'por'),
    'output_path': os.environ.get(
        'NFSE_TESSERACT_OUTPUT',
        str(Path('/content') / 'tesseract_raw_artifacts.json' if IS_COLAB else ROOT / 'artifacts' / 'tesseract_raw_artifacts.json'),
    ),
}

print(f'ROOT: {ROOT}')
print(f'IS_COLAB: {IS_COLAB}')


In [ ]:
if IS_COLAB and CONFIG['bootstrap_runtime']:
    subprocess.run(['bash', str(ROOT / 'scripts' / 'colab_bootstrap.sh')], check=True)

if IS_COLAB and CONFIG['mount_drive']:
    from google.colab import drive
    drive.mount('/content/drive')


In [ ]:
# Optional: upload a sensitive sample directly into the Colab runtime.
if IS_COLAB and not CONFIG['sample_path']:
    from google.colab import files

    uploaded = files.upload()
    if uploaded:
        first_file = next(iter(uploaded))
        CONFIG['sample_path'] = str(Path('/content') / first_file)
        print(f"Sample path: {CONFIG['sample_path']}")


In [ ]:
from pathlib import Path

from src.engines import TesseractExtractionAdapter
from src.export import write_extracted_elements_json
from src.ingestion import load_document
from src.preprocessing import preprocess_document

if not CONFIG['sample_path']:
    raise ValueError('Set CONFIG[\'sample_path\'] or upload a sample before running the pipeline.')

sample_path = Path(CONFIG['sample_path'])
if not sample_path.exists():
    raise FileNotFoundError(f'Sample file not found: {sample_path}')

document = load_document(sample_path)
preprocessed = preprocess_document(document)
adapter = TesseractExtractionAdapter(language=CONFIG['language'])
artifacts = adapter.extract_preprocessed(preprocessed)
output_path = write_extracted_elements_json(artifacts, CONFIG['output_path'])

print(f'Document: {document.document_id}')
print(f'Media type: {document.media_type}')
print(f'Pages: {preprocessed.metadata["page_count"]}')
print(f'Artifacts: {len(artifacts)}')
print(f'Output: {output_path}')


In [ ]:
for item in artifacts[:80]:
    print(
        f'{item.text!r} | conf={item.confidence} | page={item.page_number} | bbox={item.bounding_box}'
    )
